# Lecture 3 demo: ML fundamentals

Varada Kolhatkar

# Lecture 3 Class Demo

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.dummy import DummyRegressor
from sklearn.model_selection import cross_validate, train_test_split
from sklearn.tree import DecisionTreeRegressor, plot_tree
RANDOM_STATE = 123

Let's use housing prices to investigate a question from last lecture: **does a great training score mean we have a useful model?**

We'll build a first model, question how we evaluated it, and improve our workflow together. The opening steps are deliberately naive: we explore and fit on all the data before introducing splitting. By the end, we'll explain what we would do differently on a fresh project.

**How to use this notebook:** pause at the questions before running the next cell. Code cells marked `TODO` contain the worked solution for now; try describing or writing your approach before reading it. You can hide the solution while we work together.

Run cells in order in JupyterLab or VS Code. Put `kc_house_data.csv` in a `data` folder beside this notebook, and use a kernel with NumPy, pandas, Matplotlib, and scikit-learn installed.

[Chapter 3: ML fundamentals](https://ubc-cs.github.io/cpsc330-book/book/03_ml-fundamentals.html)

## The prediction problem

The King County housing dataset contains records of house sales, including sale prices and information about each property. The data are available from [House Sales in King County, USA](https://www.kaggle.com/harlfoxem/housesalesprediction). 

Imagine that we want to estimate a property's sale price from its recorded characteristics.

In [ ]:
data_path = Path("data/kc_house_data.csv")

housing_df = pd.read_csv(data_path)
housing_df

### ❓❓ Questions for you

**Discuss before coding**

- What does one example represent? What is the target?
- Is this a classification problem or a regression problem?
- What would it mean for this model to generalize?

## Exploratory Data Analysis (EDA)

Before building anything, let's get to know these house sales.

**TODO: take a first look.** How many rows and columns do we have? What do the summaries and missing-value counts tell us?

We're using the full dataset in this opening demonstration. On a fresh project, we would reserve test data **before EDA**. Keep track of this shortcut—we'll return to it at the end.

In [ ]:
# How many data points do we have? 
housing_df.shape[0]

In [ ]:
# What are the columns in the dataset? 
housing_df.columns

What ranges do the numerical features have? Let's try `describe()`. Does a numerical dtype always mean a column should be treated as a numerical measurement?

In [ ]:
housing_df.describe()

In [ ]:
# TODO: check whether any columns have missing values.
housing_df.isna().sum()


### Which columns should we use?

Let's discuss what each column represents, not just whether Python can accept it.

**Would `id` help predict a new property's price?** Compare the number of distinct IDs with the number of sales. Could a property appear more than once?

In [ ]:
housing_df['id'].unique().shape[0]

**What could we learn from `date`?** Would passing this raw string to a tree work?

In [ ]:
housing_df['date']

In [ ]:
dates = pd.to_datetime(['20141013T000000', '20141209T000000', '20150218T000000'], format='%Y%m%dT%H%M%S')
dates.month_name()

**Could location help?** A ZIP code contains location information, but does a larger code mean 'more' of something?

In [ ]:
housing_df['zipcode']

**What do you expect from `waterfront`?** Check its value counts. How common are waterfront properties?

In [ ]:
# What are the value counts of the `waterfront` feature? 
housing_df['waterfront'].value_counts()

In [ ]:
# What are the value_counts of `yr_renovated` feature? 
housing_df['yr_renovated'].value_counts()

For today, we'll omit `id`, `date`, and `zipcode` to focus on evaluation. This is a teaching simplification, not a claim that dates or location are useless. We'll revisit feature preparation later.

### Your turn: separate inputs and target

**TODO:** create `X` without the target and the three omitted columns, and create `y` from `price`. Why must `price` stay out of `X`?

In [ ]:
# TODO: try this step before reading the worked solution below.
X = housing_df.drop(columns = ['id', 'date', 'zipcode', 'price'])

In [ ]:
# TODO: try this step before reading the worked solution below.
y = housing_df['price']

## A baseline: predict the same price for every house

**Before running:** if you had to predict one price for everyone, what would you choose?

**TODO:** fit a `DummyRegressor` on `X, y`, inspect a few predictions, and score it on those same data. We have no validation set yet: this is a **training score**.

For these regressors, `.score()` returns **R², not accuracy**. Higher is better: 1 is perfect; 0 matches predicting the mean target of the evaluated data; negative values are possible. A baseline fitted on another subset need not score exactly 0 on held-out data.

In [ ]:
# TODO: try this step before reading the worked solution below.
# Train a DummyRegressor model 

from sklearn.dummy import DummyRegressor # Import DummyRegressor 

# Create a class object for the sklearn model.
dummy_regr = DummyRegressor()

# fit the dummy regressor


# score the model 



**Pause:** why is this training score approximately zero? Does that mean none of the predicted prices are useful?

In [ ]:
dummy_regr.predict(X.iloc[:5])

## Can a decision tree do better?

**Predict first:** will its training score be higher than the baseline's?

**TODO:** fit an unrestricted `DecisionTreeRegressor` on `X, y` and score it on the same examples.

In [ ]:
# TODO: try this step before reading the worked solution below.
# Train a decision tree model 

from sklearn.tree import DecisionTreeRegressor # Import DecisionTreeRegressor 

# Create a class object for the sklearn model.
dt_regr = 


# fit the decision tree regressor 


# score the model 

### Intuition of regression trees

- In classification trees, a "good split" is one that makes each group more pure in terms of labels (e.g., mostly happy 🙂 vs. mostly unhappy 🙁).
- In regression trees, a good split is one that creates groups where the target values have low variance around their mean.

See [the documentation](https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeRegressor.html) for more details. 

### Stop and discuss

Our training score looks impressive. **Would you deploy this model?**

- Which examples did the model see during fitting?
- Have we measured performance on any unseen examples?
- What evidence would make you more confident?

What's the depth of this model? 

In [ ]:
dt_regr.get_depth()

## Data splitting 

Our first score answered, 'How well can the model fit data it has already seen?' We want to ask, **'How well will it predict new examples?'**

**Slide pause:** connect training, validation, test, and deployment to practice questions, held-out practice questions, the exam, and interview questions.

**TODO:** split `X, y` into 80% training and 20% test data. Fit a **fresh** tree on training data only.

This keeps the new fit separate from the held-out rows, but does not undo our earlier full-data exploration. We'll use this split to learn the mechanics; it is not a pristine final assessment.

In [ ]:
# TODO: try this step before reading the worked solution below.
# Split the data 
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = 

In [ ]:
# Instantiate a class object 
dt = DecisionTreeRegressor(random_state=RANDOM_STATE)

# Train a decision tree on X_train, y_train


# Score on the train set


In [ ]:
# Score on the test set


### What changed?

- How do the training and held-out scores compare? What could explain the gap?
- What do we gain and lose by reserving a larger test set?
- Why fix `random_state`? Would searching for a favourable split improve the model or just its reported score?

**TODO:** also fit the dummy baseline on training data only. Compare its training and held-out scores with the tree's.

In [ ]:
# Compare with a baseline fitted only on the training subset.
dummy_split = DummyRegressor().fit(X_train, y_train)
pd.DataFrame({
    "train_score": [dummy_split.score(X_train, y_train), dt.score(X_train, y_train)],
    "held_out_score": [dummy_split.score(X_test, y_test), dt.score(X_test, y_test)],
}, index=["Dummy regressor", "Decision tree"])


## Does limiting depth help?

**Predict first:** what will happen to both scores if we allow only one split?

**TODO:** fit a decision stump and inspect its split. We're about to try model choices against the test set—watch for the evaluation problem this creates.

In [ ]:
# TODO: try this step before reading the worked solution below.
# max_depth= 1 
dt = DecisionTreeRegressor(max_depth=1, random_state=RANDOM_STATE) 
dt.fit(X_train, y_train)

In [ ]:
# Visualize your decision stump
from sklearn.tree import plot_tree 
plot_tree(dt, feature_names = X.columns.tolist(), impurity=False, filled=True, fontsize=10);

In [ ]:
dt.score(X_train, y_train) # Score on the train set

In [ ]:
dt.score(X_test, y_test) # Score on the test set

**Discuss:** are both scores poor, or is only the held-out score poor? What would support calling this underfitting?

Now allow `max_depth=20`. **Predict before running:** must both scores improve?

In [ ]:
dt = DecisionTreeRegressor(max_depth=20, random_state=RANDOM_STATE)
dt.fit(X_train, y_train)

In [ ]:
dt.score(X_train, y_train) # Score on the train set

In [ ]:
dt.score(X_test, y_test) # Score on the test set

Which depth looks more promising? **Which dataset just influenced that choice?**

**Slide pause:** underfitting, overfitting, the fundamental trade-off, and the golden rule. Use the exam analogy: learning too little versus memorizing practice-specific details.

## Single validation set

We've been looking at test scores to compare depths. That turns the test set into part of model selection: it is no longer an independent final assessment.

Let's introduce a **validation set** for further choices. This improves our workflow from here onward, but **does not make the already-used test set fresh again**.

**TODO:** reserve 20% of `X_train, y_train` for validation. What fractions of the original data are now used for fitting, validation, and testing?

In [ ]:
# TODO: try this step before reading the worked solution below.
# Create a validation set 
X_tr, X_valid, y_tr, y_valid = train_test_split(X_train, y_train, test_size=0.2, random_state=RANDOM_STATE)

In [ ]:
# TODO: try this step before reading the worked solution below.
tr_scores = []
valid_scores = []
depths = np.arange(1, 35, 2)

for depth in depths:  
    # Create and fit a decision tree model for the given depth  
    dt = DecisionTreeRegressor(max_depth=depth, random_state=RANDOM_STATE)

    dt.fit(X_tr, y_tr)
    
    # Calculate and append r2 scores on the training and validation sets
    tr_scores.append(dt.score(X_tr, y_tr))    
    valid_scores.append(dt.score(X_valid, y_valid))
    
results_single_valid_df = pd.DataFrame({"train_score": tr_scores, 
                           "valid_score": valid_scores},index = depths)
results_single_valid_df

In [ ]:
# Plot the scores collected above.
results_single_valid_df.plot(
    xlabel="Maximum tree depth",
    ylabel="R² score",
    title="Tree depth and performance on a single validation split",
)
plt.show()

### Read the plot together

- Where do you see evidence of underfitting or overfitting? Use both scores.
- Does extra depth always improve validation performance?
- Which depth would you choose?
- Would another validation split lead to the same choice?

In [ ]:
# TODO: try this step before reading the worked solution below.
# What depth gives the "best" validation score?
best_depth = results_single_valid_df['valid_score'].idxmax() 
best_depth

## Would another split tell the same story?

**Slide pause:** five-fold cross-validation. Each configuration is fitted five times, with a different fold held out each time.

**TODO:** compare the same depths with five-fold CV on `X_train, y_train`. Record mean training score, mean validation score, and validation-score standard deviation.

Before running:
- Does `test_score` in `cross_validate` mean our final test set?
- How many fits will 17 candidate depths require?
- Why use `X_train` rather than the smaller `X_tr`?

In [ ]:
depths = np.arange(1, 35, 2)
depths

In [ ]:
# TODO: compare depths using five-fold CV (worked solution below).
cv_train_scores = []
cv_valid_scores = []
cv_valid_stds = []

for depth in depths:
    dt = DecisionTreeRegressor(max_depth=depth, random_state=RANDOM_STATE)
    results = cross_validate(
        
    )
    cv_train_scores.append(results['train_score'].mean())
    # Here test_score refers to validation folds, not X_test.
    cv_valid_scores.append(results['test_score'].mean())
    cv_valid_stds.append(results['test_score'].std())

In [ ]:
cv_results_df = pd.DataFrame({
    "train_score": cv_train_scores,
    "valid_score": cv_valid_scores,
    "valid_std": cv_valid_stds,
}, index=depths)
cv_results_df.index.name = "max_depth"
cv_results_df

In [ ]:
cv_results_df[["train_score", "valid_score"]].plot(
    xlabel="Maximum tree depth",
    ylabel="Mean R² score",
    title="Tree depth and five-fold cross-validation performance",
)
plt.show()

**TODO:** find the depth with the highest mean CV score. Is a tiny improvement enough to prefer a much deeper tree?

In [ ]:
# TODO: try this step before reading the worked solution below.
best_depth = cv_results_df['valid_score'].idxmax()
best_depth

### ❓❓ Questions for you

**Discuss the following questions with your neighbour**

- How does this comparison differ from the single validation split?
- How much do validation scores vary across folds? How does that affect your confidence in small differences between depths?
- If a shallower tree scores nearly as well as the highest-scoring tree, which would you choose? Explain.
- Have we found the best possible depth, or the best choice according to a particular comparison?

## Refit the selected configuration and evaluate

**Before running:** fix your chosen depth and explain why you chose it.

**TODO:** fit a fresh tree on **all non-test training data**, then evaluate it on the test set. Why do we refit instead of keeping one of the CV models?

In a fresh workflow, this would be our first look at test performance. **Here, we already used these data during EDA and model comparisons, so this score is illustrative—not an independent final estimate.** A new independent evaluation would be needed for that claim.

In [ ]:
# TODO: record your choice before looking at the final score.
selected_depth = int(best_depth)
selection_reason = "Highest mean validation R² among the depths compared with CV."
print(f"Selected depth: {selected_depth}. {selection_reason}")

# Refit on all non-test training data.
dt_final = DecisionTreeRegressor(max_depth=selected_depth, random_state=RANDOM_STATE)
dt_final.fit(X_train, y_train)
dt_final.score(X_train, y_train)

In [ ]:
# TODO: try this step before reading the worked solution below.
# Evaluate on the test set
dt_final.score(X_test, y_test)

How does this score compare with the CV estimate? What might explain a difference?

If we now changed the depth because of this score, which role would the test set be playing?

## Optional: what features did the tree use?

In [ ]:
#What's the depth of the model? 
dt_final.get_depth()

In [ ]:
# plot_tree(dt_final, feature_names = X_train.columns.tolist(), impurity=False, filled=True);

In [ ]:
# Which features are the most important ones?
dt_final.feature_importances_

**TODO:** pair each importance with its feature name and sort the table. Which features does this tree rely on most?

These are impurity-based importances for this fitted tree. They do not establish causation; correlated features can share importance, and features with many possible splits can be favoured. If this exploration motivates another modeling choice, return to validation.

In [ ]:
# TODO: try this step before reading the worked solution below.
df = pd.DataFrame( 
    data = {
        "features": dt_final.feature_names_in_,
        "feature_importances": dt_final.feature_importances_
    }
)
df.sort_values("feature_importances", ascending=False)

## Looking back: what did we learn?

Discuss with a neighbour:

1. Why wasn't the initial training score enough?
2. When did our test set start influencing model selection?
3. Why didn't creating a validation set later undo that?
4. What did CV add beyond one validation split?
5. Where did you see underfitting or overfitting?

## The workflow to take away

On a **fresh project**:

1. Define the prediction task and reserve a test set using an appropriate split.
2. Explore the **training data** and prepare features.
3. Establish a baseline; compare models and hyperparameters using **CV on the training data**. Learn any preprocessing within each training fold.
4. Fix the configuration and **refit on all training data**.
5. Evaluate on the **untouched test set**.
6. Optionally inspect feature importances to understand the fitted model.

**The golden rule:** test data must not guide training or model selection.

Our opening shortcuts helped us discover this workflow. They are not steps to copy into a final evaluation.